In [8]:
from matplotlib.gridspec import GridSpec
import networkx as nx
from itertools import product
from collections import defaultdict
import scipy as sp
import matplotlib as mpl
from scipy.stats import circmean
from scipy.stats import circstd
from pathlib import Path
import sys
import numpy as np
import glob
import os
from scipy.stats import iqr
from scipy.ndimage import gaussian_filter


%load_ext autoreload
%autoreload

def get_parent_dir():
    try:
        return Path(__file__).resolve().parent.parent
    except NameError:
        return Path.cwd().parent

parent_dir = str(get_parent_dir())
print("Parent directory:", parent_dir)

sys.path.append(parent_dir)
from experiments_analysis.entrainment_experiments import *
from experiments_analysis.entrainment_analysis import *


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Parent directory: c:\Users\HP\PycharmProjects\Internproject 2025\cerebellum-jax-main


In [21]:
seedlist = np.arange(88, 92)
ISI_values = np.linspace(20, 200, 4)
for ISI in ISI_values:
    for seed in seedlist:
        print(f'python3 main_entrain.py --experiment specific-isi --run-type train --PFPC_plasticity-on True --OU-stim-isi-mean {ISI} --seed {seed}.txt')

python3 run_entrain.py --experiment specific-isi --run-type train --PFPC_plasticity-on True --OU-stim-isi-mean 20.0 --seed 88.txt
python3 run_entrain.py --experiment specific-isi --run-type train --PFPC_plasticity-on True --OU-stim-isi-mean 20.0 --seed 89.txt
python3 run_entrain.py --experiment specific-isi --run-type train --PFPC_plasticity-on True --OU-stim-isi-mean 20.0 --seed 90.txt
python3 run_entrain.py --experiment specific-isi --run-type train --PFPC_plasticity-on True --OU-stim-isi-mean 20.0 --seed 91.txt
python3 run_entrain.py --experiment specific-isi --run-type train --PFPC_plasticity-on True --OU-stim-isi-mean 80.0 --seed 88.txt
python3 run_entrain.py --experiment specific-isi --run-type train --PFPC_plasticity-on True --OU-stim-isi-mean 80.0 --seed 89.txt
python3 run_entrain.py --experiment specific-isi --run-type train --PFPC_plasticity-on True --OU-stim-isi-mean 80.0 --seed 90.txt
python3 run_entrain.py --experiment specific-isi --run-type train --PFPC_plasticity-on Tru

- Test saving and loading state
- Set up command generator for training
- Set up argparse

- initialize network and runner objects function (without actually running)
- run until convergence function
- snapshot state function
- restore state function

 - parse argument function
 - main (for CLI) (passes config to single training and testing functions)

 - run_single_training_session function 
 - run_single_testing_session function

 - command generator function for all training conditions

Overview run functions/contrasts I need: 

-Training:
o Specific stimulus interval for training, "OU_stim_isi_mean", "OU_stim_amp_io_mean", "OU_stim_amp_pf_mean"
o Random stimulus interval for training, "OU_stim_isi_mean" , "OU_stim_isi_std", "OU_stim_amp_io_mean", "OU_stim_amp_pf_mean"
o No stimulus training / convergence, "OU_stim_amp_pf_mean" , "OU_stim_amp_io_mean"

-Testing:
o Specific stimulus trained with different ISI, plastic

o Specific stimulus trained with different ISI, static

o Specific stimulus trained with random ISI, plastic

o Specific stimulus trained with random ISI, static

o Random stimulus trained with different ISI, plastic

o Random stimulus trained with different ISI, static

o Random stimulus trained with random ISI, plastic

o Random stimulus trained with random ISI, static


o No stimulus trained with different ISI, plastic

o No stimulus trained with different ISI, static

o No stimulus trained with random ISI, plastic

o No stimulus trained with random ISI, static



In [ ]:
# Training experiments

In [9]:
#  Test for run_train

parent_dir = get_parent_dir()

results_dir = os.path.join(
    parent_dir, "results", f"stim_experiments_test_for_train_{time.strftime('%m-%d_%H;%M;%S')}"
)
snapshot_dir = os.path.join(
    parent_dir, "states", f"state_test_for_train_{time.strftime('%m-%d_%H;%M;%S')}"
)
figures_dir = os.path.join(
    parent_dir, "figures", f"state_test_for_train_{time.strftime('%m-%d_%H;%M;%S')}"
)

os.makedirs(results_dir, exist_ok=True)
os.makedirs(snapshot_dir , exist_ok=True)
os.makedirs(figures_dir , exist_ok=True)


run_path = os.path.join(results_dir, f"test_train_runner.npz")


net_params = {"PFPC_plasticity_on": True, 
             "OU_stim_pf_on": False,
            "OU_stim_io_on": False,
}

run_params = { "seed": 88,
        "dt": 0.025,
        "downsample": 50, 
        "max_runtime": 8_000,
}

config = {
        "net_params": net_params,
        "run_params": run_params,
       
        "snapshot_dir": snapshot_dir,
        "run_path": run_path,
        "figures_dir": figures_dir,

    }

net, data, state = run_train(config)




Not converged, t=8000 ms
Simulation time taken = 29.02840781211853 s
Saving checkpoint into C:\Users\HP\PycharmProjects\Internproject 2025\cerebellum-jax-main\states\state_test_for_train_05-27_09;37;56\_state.bp
Saving time taken: 0.15207934379577637 s 


In [ ]:
# Plot weights over time
max_time = np.max(data['ts'])
print(max_time)
figure_2DEF(seed = run_params['seed'], mon= data,  save= True, save_dir = config['figures_dir'],   threshold = 2, xrange = [0.001, max_time-1])

In [12]:
print(state.items())


dict_items([('CerebellarNetwork3', {}), ('PFBundles3', {'PFBundles3.I_OU': Array([1.2258683, 0.8519133, 0.8873093, 1.7224342, 1.060859 ], dtype=float32), 'PFBundles3.rho': Array([0.07413161, 0.44808668, 0.41269064, 0.4224342 , 0.23914099],      dtype=float32), 'PFBundles3.I_stim': Array([0., 0., 0., 0., 0.], dtype=float32)}), ('PurkinjeCell3', {'PurkinjeCell3.V': Array([-58.50077 , -56.600273, -51.23742 , -55.692642, -55.66595 ,
       -55.42416 , -51.670155, -54.212215, -57.10386 , -68.10723 ,
       -56.74092 , -53.20556 , -54.791576, -54.37496 , -55.6461  ,
       -53.2466  , -50.62911 , -54.352684, -56.144375, -55.784573,
       -56.364334, -54.172005, -51.95771 , -55.441967, -52.94843 ,
       -50.58377 , -52.44963 , -54.5076  , -55.821598, -53.810707,
       -55.009968, -53.139412, -49.231564, -55.31093 , -52.721046,
       -52.349644, -53.03814 , -56.975   , -60.800713, -54.469505,
       -51.311806, -57.289497, -61.81265 , -60.023   , -55.763897,
       -49.631275, -58.51944 , 

In [17]:
# Test for restoring network

parent_dir = get_parent_dir()
results_dir = os.path.join(
    parent_dir, "results", f"stim_experiments_test_for_test_{time.strftime('%m-%d_%H;%M;%S')}"
)
run_path = os.path.join(results_dir, f"test_test_runner.npz")

net_params = {"PFPC_plasticity_on": False, 
             "OU_stim_pf_on": True,
            "OU_stim_io_on": True,
}

run_params = { "seed": 88,
        "dt": 0.025,
        "simdur": 100_000,
        "downsample": 50, 
}

config = {
    'net_params': net_params,
    'run_params': run_params,
    'pretrain_snapshot_dir': snapshot_dir,
    'run_path':  run_path,

}

## Within run_test
new_net, new_runner = init_net_and_runner(net_params, seed= run_params['seed'])
print("Before - :", new_net.vars())

state, meta = load_snapshot(config['pretrain_snapshot_dir'])
bp.load_state(new_net, state)
print("After - :", new_net.vars())



print(state.items())


Before - : {'PFBundles8.I_OU': Variable(
  value=ShapedArray(float32[5]),
  _batch_axis=None,
  axis_names=None
), 'PFBundles8.rho': Variable(
  value=ShapedArray(float32[5]),
  _batch_axis=None,
  axis_names=None
), 'PFBundles8.I_stim': Variable(
  value=ShapedArray(float32[5]),
  _batch_axis=None,
  axis_names=None
), 'PurkinjeCell8.V': Variable(
  value=ShapedArray(float32[100]),
  _batch_axis=None,
  axis_names=None
), 'PurkinjeCell8.w': Variable(
  value=ShapedArray(float32[100]),
  _batch_axis=None,
  axis_names=None
), 'PurkinjeCell8.input': Variable(
  value=ShapedArray(float32[100]),
  _batch_axis=None,
  axis_names=None
), 'PurkinjeCell8.dbg_leak': Variable(
  value=ShapedArray(float32[100]),
  _batch_axis=None,
  axis_names=None
), 'PurkinjeCell8.dbg_exp': Variable(
  value=ShapedArray(float32[100]),
  _batch_axis=None,
  axis_names=None
), 'PurkinjeCell8.dbg_current': Variable(
  value=ShapedArray(float32[100]),
  _batch_axis=None,
  axis_names=None
), 'PurkinjeCell8.dbg_w'

KeyError: 'CerebellarNetwork8'

In [ ]:
print(data.keys())
print(snapshot_dir)



In [ ]:
# Load baseline run datasets
datasets = [
    r'C:\Users\HP\Modelling projects\Olivocerebellar circuit\results\stim_experiments_nostim_06-11_12;57;34\train_nostim_isi120.0_seed88.npz',
    r'C:\Users\HP\Modelling projects\Olivocerebellar circuit\results\stim_experiments_nostim_06-11_12;57;58\train_nostim_isi120.0_seed89.npz',
    r'C:\Users\HP\Modelling projects\Olivocerebellar circuit\results\stim_experiments_nostim_06-11_12;58;23\train_nostim_isi120.0_seed90.npz',
    r'C:\Users\HP\Modelling projects\Olivocerebellar circuit\results\stim_experiments_nostim_06-11_12;58;48\train_nostim_isi120.0_seed91.npz',
    ]


# Load weight datasets from runners. Get the weights for each chunk.
chunk_size = 1000 # ms


